# BiomedParse Fascia Fine-Tuning — Production v1

Run cells **in order**. Training takes ~2 h on RTX 5090 (10 epochs).

## Why v5 produced IoU=0 and why v1 fixes it

| Problem | v5 (broken) | v1 (fixed) |
|---------|-------------|------------|
| Training data | 2364 fascia **+ 1731 vein** | **fascia only** |
| Learning rate | 1e-6 (barely moves weights) | **1e-4** |
| Epochs | 5 | **10** |
| pixel_decoder | unfrozen (too many params) | **frozen** |
| Inference query selection | VL-similarity → dead queries | **max-logit** |
| Training dice=0.96 | oracle bipartite match (fake) | oracle match; real IoU checked in eval cells below |

**Root cause of IoU=0**: the model learned to detect veins (easy circular blobs, 1731 annotations).  
At LR=1e-6 the fascia loss was never strong enough to overcome the vein gradient.  
Oracle dice was high because bipartite matching always finds the best of 101 queries —  
that single best query happened to be vein, not fascia.

---
## 1 · Setup

In [ ]:
import os
import sys
import json
import shutil
import subprocess
from pathlib import Path

_NB_DIR     = Path(os.path.abspath(''))
_BASE_DIR   = _NB_DIR
BIOMEDPARSE = _BASE_DIR / 'BiomedParse'
FASCIA_DATA = BIOMEDPARSE / 'biomedparse_datasets' / 'Fascia_Detection'

sys.path.insert(0, str(_BASE_DIR / 'stubs'))
sys.path.insert(0, str(BIOMEDPARSE))

print(f'Base dir:    {_BASE_DIR}')
print(f'BiomedParse: {BIOMEDPARSE}')
print(f'Fascia data: {FASCIA_DATA}')
assert FASCIA_DATA.exists(), f'Dataset dir not found: {FASCIA_DATA}'
assert (FASCIA_DATA / 'train.json').exists(), 'train.json missing'
assert (FASCIA_DATA / 'test.json').exists(),  'test.json missing'
print('All paths OK.')

---
## 2 · Filter training data to fascia-only

The original JSONs contain **category_id 17 (fascia)** and **category_id 18 (vein)**.  
We remove all vein annotations before training so the model only learns fascia.  
Originals are backed up as `train_with_veins_backup.json` / `test_with_veins_backup.json`.

In [ ]:
def filter_fascia_only(json_path: Path, backup_suffix='_with_veins_backup') -> dict:
    """Filter annotations to category_id=17 (fascia). Backs up original first."""
    backup = json_path.with_name(json_path.stem + backup_suffix + json_path.suffix)
    if not backup.exists():
        shutil.copy(json_path, backup)
        print(f'  Backed up: {json_path.name} -> {backup.name}')
    else:
        print(f'  Backup already exists: {backup.name}')

    with open(backup) as f:
        data = json.load(f)

    all_anns    = data['annotations']
    fascia_anns = [a for a in all_anns if a['category_id'] == 17]
    vein_anns   = [a for a in all_anns if a['category_id'] == 18]

    filtered = dict(data)
    filtered['annotations'] = fascia_anns

    with open(json_path, 'w') as f:
        json.dump(filtered, f)

    print(f'  {json_path.name}: {len(all_anns)} total -> {len(fascia_anns)} fascia (removed {len(vein_anns)} vein)')
    return filtered


print('Filtering train.json ...')
train_data = filter_fascia_only(FASCIA_DATA / 'train.json')

print('Filtering test.json ...')
test_data  = filter_fascia_only(FASCIA_DATA / 'test.json')

print()
print(f'Training on {len(train_data["annotations"])} fascia annotations')
print(f'Testing on  {len(test_data["annotations"])}  fascia annotations')

---
## 3 · Train

**Key overrides vs v5:**
- `SOLVER.BASE_LR 0.0001` — 100× higher than the broken 1e-6
- `SOLVER.MAX_NUM_EPOCHS 10` — double the epochs
- `SOLVER.FIX_PARAM.pixel_decoder True` — keep pixel decoder frozen (v5 had this `False` by mistake)
- `SAVE_DIR output/fascia_finetuning_v1_production`

In [ ]:
env = dict(os.environ)
env['DETECTRON2_DATASETS'] = str(BIOMEDPARSE / 'biomedparse_datasets')
env['DATASET']             = str(BIOMEDPARSE / 'biomedparse_datasets')
env['DATASET2']            = str(BIOMEDPARSE / 'biomedparse_datasets')
env['VLDATASET']           = str(BIOMEDPARSE / 'biomedparse_datasets')
env['PYTHONPATH']          = (
    str(_BASE_DIR / 'stubs') + os.pathsep +
    str(BIOMEDPARSE) + os.pathsep +
    env.get('PYTHONPATH', '')
)
env['PYTHONUTF8'] = '1'

cmd = [
    sys.executable, '-u', 'entry.py', 'train',
    '--conf_files', 'configs/biomed_fascia_finetuning.yaml',
    '--overrides',
    'FP16',                          'True',
    'RANDOM_SEED',                   '2024',
    'BioMed.INPUT.IMAGE_SIZE',       '512',
    'TRAIN.BATCH_SIZE_TOTAL',        '4',
    'TRAIN.BATCH_SIZE_PER_GPU',      '4',
    'SOLVER.MAX_NUM_EPOCHS',         '10',
    'SOLVER.BASE_LR',                '0.0001',
    'SOLVER.FIX_PARAM.backbone',     'True',
    'SOLVER.FIX_PARAM.lang_encoder', 'True',
    'SOLVER.FIX_PARAM.pixel_decoder','True',
    'WEIGHT',                        'True',
    'LOG_EVERY',                     '1',
    'RESUME_FROM',    str(_BASE_DIR / 'pretrained' / 'biomedparse_v1.pt'),
    'SAVE_DIR',       'output/fascia_finetuning_v2_production',   # v1 had empty GT masks (mapper bug); v2 is fixed
]

print('=' * 60)
print('FASCIA-ONLY | LR=1e-4 | 10 epochs | pixel_decoder FROZEN')
print('Mapper fix: m==3 (was m==1, gave empty GT every time)')
print('Checkpoints -> BiomedParse/output/fascia_finetuning_v2_production/')
print('=' * 60, flush=True)

proc = subprocess.Popen(
    cmd,
    cwd=str(BIOMEDPARSE),
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    encoding='utf-8',
    errors='replace',
)

for line in proc.stdout:
    print(line, end='', flush=True)

proc.wait()
print(f'\nDone -- exit code {proc.returncode}')


---
## 4 · Load fine-tuned model

**Critical**: load with `biomed_fascia_finetuning.yaml`, NOT `biomedparse_inference.yaml`.  
The inference YAML has `CROSS_ATTENTION.queries = {object: true}` only — missing `grounding: true`.  
Without `grounding: true`, `pred_gmasks` is never populated → KeyError at inference.

In [ ]:
import glob as _glob
import torch
import numpy as np
import os
import torch.nn.functional as F
from PIL import Image
from pathlib import Path

# Re-derive paths (safe to run standalone without setup cell)
_BASE_DIR   = Path(os.path.abspath(''))
if _BASE_DIR.name == 'BiomedParse':
    _BASE_DIR = _BASE_DIR.parent
BIOMEDPARSE = _BASE_DIR / 'BiomedParse'
FASCIA_DATA = BIOMEDPARSE / 'biomedparse_datasets' / 'Fascia_Detection'

import sys
sys.path.insert(0, str(_BASE_DIR / 'stubs'))
sys.path.insert(0, str(BIOMEDPARSE))

os.chdir(str(BIOMEDPARSE))

from modeling.BaseModel import BaseModel
from modeling import build_model
from utilities.distributed import init_distributed
from utilities.arguments import load_opt_from_config_files
from detectron2.structures import ImageList

# v2 = first run with correct GT masks (mapper bug fixed: m==3 not m==1)
_ckpt_dir = BIOMEDPARSE / 'output' / 'fascia_finetuning_v2_production'
_ckpts = sorted(
    _glob.glob(str(_ckpt_dir / '**' / 'model_state_dict.pt'), recursive=True),
    key=os.path.getmtime
)
assert _ckpts, f'No checkpoint in {_ckpt_dir} — run the training cell first.'
FINETUNED_WEIGHTS = _ckpts[-1]
print(f'Checkpoint: {FINETUNED_WEIGHTS}')

# TRAINING config — has CROSS_ATTENTION.queries.grounding: True
opt = load_opt_from_config_files([str(BIOMEDPARSE / 'configs' / 'biomed_fascia_finetuning.yaml')])
opt = init_distributed(opt)

model = BaseModel(opt, build_model(opt)).from_pretrained(FINETUNED_WEIGHTS).eval().cuda()
with torch.no_grad():
    model.model.sem_seg_head.predictor.lang_encoder.get_text_embeddings(
        ['fascia layer', 'background'], is_eval=True
    )

print(f'Model class:  {type(model.model).__name__}')
print(f'task_switch:  {model.model.task_switch}')
print('Model loaded OK.')


---
## 5 · Inference functions

- `infer_grounding`: calls predictor directly (bypasses `evaluate_demo` which doesn't exist on `GeneralizedSEEM`)
- `grounding_to_prob`: picks best query by **max-logit** (NOT VL-similarity — VL-sim picks dead queries after fine-tuning)

In [ ]:
def infer_grounding(pil_image, text='fascia layer in PeripheralVascular Ultrasound', infer_size=512):
    """
    Production fascia grounding inference.
    Calls predictor directly with task='grounding_eval'.
    Returns (outputs, extra)  where outputs['pred_gmasks'] has shape [1, 101, H', W'].
    """
    m    = model.model
    pred = m.sem_seg_head.predictor

    arr   = np.asarray(pil_image.resize((infer_size, infer_size), Image.BICUBIC)).astype(np.float32)
    img_t = torch.from_numpy(arr.copy()).permute(2, 0, 1).cuda()
    images = ImageList.from_tensors(
        [(img_t - m.pixel_mean) / m.pixel_std],
        m.size_divisibility
    )

    gtext    = pred.lang_encoder.get_text_token_embeddings(
        [text], name='grounding', token=False, norm=False
    )
    tok_emb  = gtext['token_emb']
    tok_mask = gtext['tokens']['attention_mask'].bool()
    q_emb    = tok_emb[tok_mask]                                     # [T, D]
    nz_mask  = torch.zeros(q_emb[:, None].shape[:-1], dtype=torch.bool, device=q_emb.device)

    extra = {
        'grounding_tokens':       q_emb[:, None],                    # [T, 1, D]
        'grounding_nonzero_mask': nz_mask.t(),                       # [1, T]
        'grounding_class':        gtext['class_emb'],
    }

    with torch.no_grad():
        feats           = m.backbone(images.tensor)
        mf, _, ms       = m.sem_seg_head.pixel_decoder.forward_features(feats)
        outputs         = pred(ms, mf, extra=extra, task='grounding_eval')

    return outputs, extra


def grounding_to_prob(outputs, H, W, n_queries=101):
    """
    Select best grounding query by max-logit and return a probability map (H x W, float32).
    Max-logit is reliable; VL-similarity collapses to near-zero after fine-tuning.
    """
    all_gm = outputs['pred_gmasks'][0]                               # [101, H', W']
    best_q = all_gm.reshape(n_queries, -1).max(dim=1).values.argmax().item()
    raw    = F.interpolate(
        all_gm[best_q:best_q + 1][None], (H, W),
        mode='bilinear', align_corners=False
    )[0, 0]
    return torch.sigmoid(raw).cpu().numpy().astype(np.float32)


print('infer_grounding() OK')
print('grounding_to_prob() OK')

---
## 6 · Evaluate on training images

GT mask pixels: value `1` = fascia.  
Evaluates first 100 by default — change `[:100]` to `[:]` for the full set.

In [ ]:
TRAIN_IMG_DIR  = FASCIA_DATA / 'train'
TRAIN_MASK_DIR = FASCIA_DATA / 'train_mask'

with open(FASCIA_DATA / 'train.json') as f:
    train_anns = [a for a in json.load(f)['annotations'] if a['category_id'] == 17]

print(f'Evaluating {min(100, len(train_anns))} / {len(train_anns)} train images ...')

ious, dices, skipped = [], [], 0

for ann in train_anns[:100]:
    img_path  = TRAIN_IMG_DIR  / ann['file_name']
    mask_path = TRAIN_MASK_DIR / ann['mask_file']
    if not img_path.exists() or not mask_path.exists():
        skipped += 1
        continue

    pil_img = Image.open(img_path).convert('RGB')
    H, W    = pil_img.size[1], pil_img.size[0]

    gt_raw = np.array(Image.open(mask_path).convert('L'))
    gt_bin = (gt_raw == 1).astype(np.uint8)
    if gt_bin.sum() == 0:
        skipped += 1
        continue

    outputs, _ = infer_grounding(pil_img)
    prob       = grounding_to_prob(outputs, H, W)
    pred_bin   = (prob > 0.5).astype(np.uint8)

    inter = int((pred_bin & gt_bin).sum())
    union = int((pred_bin | gt_bin).sum())
    denom = int(pred_bin.sum() + gt_bin.sum())
    ious.append(inter / (union + 1e-6))
    dices.append(2 * inter / (denom + 1e-6))

print(f'Train (n={len(ious)}, skipped={skipped}):')
print(f'  mean IoU  = {np.mean(ious):.4f}')
print(f'  mean Dice = {np.mean(dices):.4f}')


---
## 7 · Evaluate on test images

In [ ]:
TEST_IMG_DIR  = FASCIA_DATA / 'test'
TEST_MASK_DIR = FASCIA_DATA / 'test_mask'

with open(FASCIA_DATA / 'test.json') as f:
    test_anns = [a for a in json.load(f)['annotations'] if a['category_id'] == 17]

print(f'Evaluating {len(test_anns)} test images ...')

ious_t, dices_t, skipped_t = [], [], 0

for ann in test_anns:
    img_path  = TEST_IMG_DIR  / ann['file_name']
    mask_path = TEST_MASK_DIR / ann['mask_file']
    if not img_path.exists() or not mask_path.exists():
        skipped_t += 1
        continue

    pil_img = Image.open(img_path).convert('RGB')
    H, W    = pil_img.size[1], pil_img.size[0]

    gt_raw = np.array(Image.open(mask_path).convert('L'))
    gt_bin = (gt_raw == 1).astype(np.uint8)
    if gt_bin.sum() == 0:
        skipped_t += 1
        continue

    outputs, _ = infer_grounding(pil_img)
    prob       = grounding_to_prob(outputs, H, W)
    pred_bin   = (prob > 0.5).astype(np.uint8)

    inter = int((pred_bin & gt_bin).sum())
    union = int((pred_bin | gt_bin).sum())
    denom = int(pred_bin.sum() + gt_bin.sum())
    ious_t.append(inter / (union + 1e-6))
    dices_t.append(2 * inter / (denom + 1e-6))

print(f'Test (n={len(ious_t)}, skipped={skipped_t}):')
print(f'  mean IoU  = {np.mean(ious_t):.4f}')
print(f'  mean Dice = {np.mean(dices_t):.4f}')


---
## 8 · Visual sanity check

Change `idx` to inspect different test images.

In [ ]:
import matplotlib.pyplot as plt

idx = 0   # change to look at different examples

ann     = test_anns[idx]
pil_img = Image.open(TEST_IMG_DIR  / ann['file_name']).convert('RGB')
H, W    = pil_img.size[1], pil_img.size[0]
gt_bin  = (np.array(Image.open(TEST_MASK_DIR / ann['mask_file']).convert('L')) == 1).astype(np.uint8)

outputs, _ = infer_grounding(pil_img)
prob       = grounding_to_prob(outputs, H, W)
pred_bin   = (prob > 0.5).astype(np.uint8)

inter = int((pred_bin & gt_bin).sum())
union = int((pred_bin | gt_bin).sum())
iou   = inter / (union + 1e-6)

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
axes[0].imshow(pil_img);                              axes[0].set_title('Input')
axes[1].imshow(gt_bin,   cmap='gray');                axes[1].set_title('GT fascia (value==1)')
axes[2].imshow(prob,     cmap='jet', vmin=0, vmax=1); axes[2].set_title('Probability map')
axes[3].imshow(pred_bin, cmap='gray');                axes[3].set_title(f'Prediction  IoU={iou:.3f}')
for ax in axes: ax.axis('off')
plt.tight_layout()
plt.show()
print(f'IoU = {iou:.4f}  |  idx={idx}  |  file={ann["file_name"]}')
